# Basic interaction with OpenAI GPT models

This example demonstrates how to set up an OpenAI GPT model and send a basic user prompt using a Jypiter notebook.

## 1. Setup

To use the OpenAI API, you need to have an API key:

1. Create or log into your [OpenAI account](https://platform.openai.com/)
2. Go to [API Keys](https://platform.openai.com/api-keys)
3. Click "Create new secret key", name it, copy it immediately
4. Add billing/payment details so the key becomes active (the API is pay-per-use)

Once you have the API key, you need to export its value as and environment variable called `OPENAI_API_KEY`. If this env is not defined, you will be asked for your API key (it will not be stored in the notebook).

In [5]:
!pip install openai

import os
import time
import re
from getpass import getpass
from openai import OpenAI

# Ask for the API key interactively if not defined as env variable
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

## 2. Function to query the model

This section defines a helper function to send a user prompt to the model.

In [ ]:
def query_model(prompt: str,
                model: str = "gpt-4o-mini",
                max_tokens: int = 1024,
                temperature: float = 0,
                reasoning: str = "low") -> str:
    """Send a user prompt to an OpenAI model and return the text response."""
    params = {
        "model": model,
        "input": prompt,
        "max_output_tokens": max_tokens,
    }
    if is_gpt5_or_above(model):
        params["reasoning"] = {"effort": reasoning}
    else:
        params["temperature"] = temperature

    start = time.perf_counter()
    response = client.responses.create(**params)
    latency = time.perf_counter() - start

    # Log some details about the response
    usage = response.usage
    print(f"\tModel: {response.model}")
    print(f"\tLatency: {latency:.3f} seconds")
    print(f"\tInput tokens: {usage.input_tokens}")
    print(f"\tOutput tokens: {usage.output_tokens}")
    print(f"\tReasoning tokens: {usage.output_tokens_details.reasoning_tokens}")
    print(f"\tTotal tokens: {usage.total_tokens}")

    return response.output_text

def is_gpt5_or_above(model: str) -> bool:
    return re.match(r"gpt-[5-9].*", model, re.IGNORECASE) is not None

## 3. Send user prompt

This section sends and user prompt to the model using the function previously defined.

In [ ]:
prompt = "How many tokens is your context window?"

print("=== Basic model  ===")
print("User:", prompt)
response = query_model(prompt)
print("GPT4:", response)
print()
print("=== Advanced model  ===")
print("User:", prompt)
response = query_model(prompt, model="gpt-5", reasoning="medium")
print("GPT5:", response)

=== Basic model  ===
User: How many tokens is your context window?
	Model: gpt-4o-mini-2024-07-18
	Latency: 2.149 seconds
	Input tokens: 15
	Output tokens: 39
	Reasoning tokens: 0
	Total tokens: 54
GPT4: My context window can handle up to 8,192 tokens. This includes both the input and the output tokens. If you have any specific questions or need assistance, feel free to ask!

=== Advanced model  ===
User: How many tokens is your context window?
	Model: gpt-5-2025-08-07
	Latency: 8.556 seconds
	Input tokens: 14
	Output tokens: 583
	Reasoning tokens: 512
	Total tokens: 597
GPT5: About 128,000 tokens (combined input + output).
